# Waveform plots — Flow & Load papers

**Usage** : set `PAPER`, `PARADIGM`, `MEASURE`, `LOCK` in Cell 1, then either  
- run **Cell 6 (Batch)** to generate all pre-configured plots, or  
- use **Cell 7 (Live)** for custom / exploratory plots during meetings.


`TRIALS` (stimulus-locked only): `"correct"` (primary) or `"alltrials"` — selects `grand_averages/{paper}/raw|diff/{TRIALS}/` and is added to the figure file names. Response-locked pkls (hits only) are read from `raw/`.

Last edit: 31.10.2026 @anguyen

In [ ]:
from __future__ import annotations
import sys
from datetime import datetime
from pathlib import Path

def find_repo_root(marker="core") -> Path:
    """Walk up from the current directory until a folder containing
    `marker` (e.g. core/) is found -- avoids hardcoding folder depth,
    which breaks every time notebooks get moved."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not locate repo root (no '{marker}/' found above {Path.cwd()})")

sys.path.insert(0, str(find_repo_root()))
from core import config_flow, config_load
from core.participants import get_participants
from core.utils import get_cluster_label, load_pkl, show_latest_pkls, tuple_to_string_fname, tuple_to_string_title

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import mne
from mne.viz import set_browser_backend

%matplotlib qt
set_browser_backend("qt")

plt.rcParams.update({
    "font.size":        12,
    "axes.titlesize":   13,
    "axes.labelsize":   12,
    "xtick.labelsize":  10,
    "ytick.labelsize":  10,
    "legend.fontsize":  10,
    "figure.titlesize": 16,
})


In [ ]:
# ==============================================================================
# Configuration — edit here, then re-run from Cell 3 downward
# ==============================================================================

PAPER:    str = "flow"
PARADIGM: str = "passive"      # "active" | "passive" / "active-n35" 
MEASURE:  str = "diff"         # "raw" | "diff"
LOCK:     str = "stim"        # "stim" | "resp"
TRIALS:   str = "correct"     # "correct" (primary) | "alltrials" — stimulus-locked only

CSD:           bool = False
BASELINE_CORR: str  = "bc-200-100"

assert TRIALS in ("correct", "alltrials"), f"Unknown TRIALS: {TRIALS}"
print(f"Paper: {PAPER.upper()} | Paradigm: {PARADIGM.upper()} | Measure: {MEASURE.upper()} | Lock: {LOCK.upper()} | Trials: {TRIALS if LOCK == 'stim' else 'n/a (hits only)'}")

# Minimal derived — needed for show_latest_pkls
PARADIGM_REAL = PARADIGM.split("-")[0]
cohort        = PARADIGM.split("-", 1)[1] if "-" in PARADIGM else None
STUDY         = "Active" if PARADIGM_REAL == "active" else "Passive"

if sys.platform == "win32":
    root = Path(f"L:/EEG_Flow_data/Study_EEG_Flow_{STUDY}/derivatives_v2")
else:
    root = Path(f"/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_{STUDY}/derivatives_v2")
    
    
if PAPER == 'flow' and LOCK == 'resp':
    raise ValueError(
        "Flow response-locked analyses use CSD and are handled separately "
        "in the Castellar notebooks (flow_paper/notebooks/). "
        "Set LOCK='stim' for flow paper plots here."
    )

# ==============================================================================
# DATE_SOURCE — None = latest pkl in the input folder (found in the next cell)
# ==============================================================================
DATE_SOURCE = None
# Override if needed:
# DATE_SOURCE = "20261031"


In [ ]:
# ==============================================================================
# Derived labels and paths (auto — do not edit)
# ==============================================================================

_cfg = config_flow if PAPER == "flow" else config_load

CONDITIONS: dict = _cfg.CONDITIONS
STYLES:     dict = _cfg.STYLES
BATCH:      dict = _cfg.BATCH

DEVIANT_LABEL = "target" if PARADIGM_REAL == "active" else "deviant"
label_csd     = "CSD" if CSD else "NOCSD"
DATE_STR      = datetime.today().strftime("%Y%m%d")

subfolder = "diff" if MEASURE == "diff" else "raw"

ga_base = (
    root / "outputs" / "grand_averages" / PAPER / "_other_cohorts" / cohort
    if cohort else
    root / "outputs" / "grand_averages" / PAPER
)
# stimulus-locked pkls live in raw/{TRIALS}/ or diff/{TRIALS}/;
# response-locked pkls (hits only, one variant) stay directly in raw/
input_path = ga_base / subfolder / TRIALS if LOCK == "stim" else ga_base / "raw"

def latest_date(folder: Path, pattern: str) -> str:
    """Most recent date prefix among pkl files matching `pattern` in `folder`."""
    files = sorted(folder.glob(pattern))
    assert files, f"No pkl matching {pattern} in {folder}"
    dates = sorted({f.name.split("_")[0] for f in files})
    print(f"Available dates: {dates}")
    return dates[-1]

if DATE_SOURCE is None:
    _pattern = (f"*_{PARADIGM}_*p_cond-*_resp_{BASELINE_CORR}_*_raw.pkl" if LOCK == "resp"
                else f"*_{PARADIGM}_*p_cond-*_stim_{subfolder}.pkl")
    DATE_SOURCE = latest_date(input_path, _pattern)
print(f"DATE_SOURCE: {DATE_SOURCE}")

root_img = (
    root / "outputs" / "img" / PAPER / "_other_cohorts" / cohort
    if cohort else
    root / "outputs" / "img" / PAPER
)
root_img.mkdir(parents=True, exist_ok=True)
(root_img / MEASURE).mkdir(parents=True, exist_ok=True)

print(f"Input  : {input_path}")
print(f"Output : {root_img}")


In [ ]:
# ==============================================================================
# Load pkl files (one per condition)
# ==============================================================================

# N_PARTICIPANTS = {
#     "flow":  {"active": 36, "passive": 41},
#     "load":  {"active": 48, "passive": 47},
# }
# n = N_PARTICIPANTS[PAPER][PARADIGM]
n = len(get_participants(PAPER, PARADIGM))


def _build_fname(cond_label: str) -> str:
    base = f"{DATE_SOURCE}_{PARADIGM}_{n}p_cond-{cond_label}"
    if LOCK == "resp":
        return f"{base}_resp_{BASELINE_CORR}_{label_csd}_raw.pkl"
    elif MEASURE == "diff":
        return f"{base}_stim_diff.pkl"
    else:
        return f"{base}_stim_raw.pkl"


evokeds: dict[str, dict] = {}   # {plot_label: {event_key: [Evoked, ...]}}

for plot_label, cond_key in CONDITIONS.items():
    fname = _build_fname(cond_key)
    evokeds[plot_label] = load_pkl(input_path / fname)
    print(f"Loaded [{plot_label}] ← {fname}")


In [ ]:
# ==============================================================================
# plot_compare_condition — core plot function
# ==============================================================================

def plot_compare_condition(
    stim: str,
    component: str,
    cluster: list[str],
    windows: tuple,
    ylim: dict,
) -> None:
    """Plot waveform comparison across conditions for one stimulus type.

    Parameters
    ----------
    stim : str
        Event type key in the pkl, e.g. "standard", "target", "target-standard".
    component : str
        ERP component label for filename and title, e.g. "N1", "P3".
    cluster : list of str
        Electrode names to average over.
    windows : tuple
        Time window(s) to highlight, e.g. ((90, 130),) or ((90,130),(200,350)).
        Pass () for no highlights.
    ylim : dict
        Y-axis limits, e.g. dict(eeg=[-8, 8]).
    """
    # --- Data loading ---
    evokeds_plot = {
        label: data[f"{CONDITIONS[label]}_{stim}"]
        for label, data in evokeds.items()
        if f"{CONDITIONS[label]}_{stim}" in data
    }
    if not evokeds_plot:
        print(f"No data found for stim='{stim}' — skipping.")
        return

    # pick EEG only — ensures consistent channel count across cohorts
    # and drops game/EOG channels not used in waveform plots
    evokeds_plot = {
        label: [e.copy().pick("eeg") for e in evo_list]
        for label, evo_list in evokeds_plot.items()
    }

    tmax = 0.300 if LOCK == "resp" else None

    evokeds_plot = {
    label: [e.copy().pick("eeg") for e in evo_list]
    for label, evo_list in evokeds_plot.items()
    }

    # --- Labels ---
    n_participants = len(next(iter(evokeds_plot.values())))
    txt_cluster    = get_cluster_label(cluster)
    fname_windows  = tuple_to_string_fname(windows)
    title_windows  = tuple_to_string_title(windows)

    # --- Plot ---
    fig, ax = plt.subplots(1, 1, figsize=(4.5, 4.5))
    mne.viz.plot_compare_evokeds(
        evokeds_plot,
        invert_y=True,
        ylim=ylim,
        axes=ax,
        vlines=[],
        truncate_yaxis=False,
        truncate_xaxis=False,
        picks=cluster,
        combine="mean",
        title="",
        styles=STYLES,
        time_unit="ms",
    )

    # --- Resp-specific: xlim + peak markers ---
    if LOCK == "resp":
        ax.set_xlim(right=300)
        for label, evo_list in evokeds_plot.items():
            ga = mne.grand_average(evo_list)
            ch_indices = [ga.ch_names.index(ch) for ch in cluster if ch in ga.ch_names]
            data = ga.data[ch_indices, :].mean(axis=0)
            times = ga.times * 1000
            mask = times <= 300
            peak_idx  = data[mask].argmax() # or argmin if peak is negative
            peak_time = times[mask][peak_idx]
            peak_val  = data[mask][peak_idx] * 1e6
            color = STYLES[label]["color"]
            ax.axvline(peak_time, color=color, linestyle="dotted", linewidth=1.2, alpha=0.7)
            ax.plot(peak_time, -peak_val, marker="v", color=color, markersize=7, zorder=5)
            ax.text(
                peak_time + 4, -peak_val,
                f"{peak_time:.0f} ms\n{peak_val:.2f} µV",
                color=color, fontsize=8, va="center",
            )

    # --- Axis decorations ---
    ax.axvline(0, color="black", linestyle="solid", linewidth=1.2)
    ax.axhline(0, color="black", linestyle="solid", linewidth=1.2)

    WINDOW_COLORS = ['#ffbd66', '#a8d8ea']

    if windows:
        if isinstance(windows[0], (int, float)):
            windows = (windows,)
        for i, (start, end) in enumerate(windows):
            ax.axvspan(start, end, facecolor=WINDOW_COLORS[i % len(WINDOW_COLORS)], alpha=0.3)

    ax.xaxis.set_major_locator(ticker.MultipleLocator(100))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(2))
    ax.grid(color="gray", linestyle="--", linewidth=0.5, alpha=0.3)
    ax.set_ylabel("Amplitude (µV)")
    ax.legend(loc="lower left")
    plt.subplots_adjust(left=0.05, right=0.95)

    # --- Titles ---
    if LOCK == "stim":
        fig.suptitle(
            f"[{PARADIGM.capitalize()}] {stim.capitalize()} · {component}"
        )
    elif LOCK == "resp":
        fig.suptitle(
            f"[{PARADIGM.capitalize()}] {stim.capitalize()} ({BASELINE_CORR})"
        )
    fig.text(
        0.5, 0.895,
        f"{title_windows} · {txt_cluster} cluster · N={n_participants}",
        ha="center", fontsize=12
    )

    bc_label_fname = f"_{BASELINE_CORR}" if LOCK == "resp" else ""
    component_fname = f"_{component}" if LOCK == "stim" else ""
    trials_fname    = f"_{TRIALS}" if LOCK == "stim" else ""
    # --- Save ---
    out_fname = (
        root_img
        / MEASURE
        / f"{PARADIGM}_{n_participants}p_{PAPER}_{MEASURE}_wave"
          f"_{stim}{component_fname}_{txt_cluster}_{fname_windows}{bc_label_fname}{trials_fname}_{DATE_STR}.svg"
    )
    print(out_fname)
    plt.savefig(out_fname, bbox_inches="tight") # dpi=300 not relevant for .svg
    plt.show()
print("plot_compare_condition() ready.")


## Batch run

Runs all pre-configured plots for the current `(PARADIGM, MEASURE)` combination.  
Edit your configs in `config_flow.py` or `config_load.py`.


In [ ]:
batch_key = (PARADIGM_REAL, LOCK) if LOCK == "resp" else (PARADIGM_REAL, MEASURE)
batch_configs = BATCH.get(batch_key, [])
if batch_configs:
    print(f"Running {len(batch_configs)} plots for {batch_key}...\n")
    for cfg in batch_configs:
        plot_compare_condition(**cfg)
else:
    print(f"No batch configs found for ({PARADIGM_REAL}, {MEASURE}).")
    print("Add entries to BATCH in config_flow.py / config_load.py,")
    print("or use the live cell below.")


## Live / custom plots

Use this cell during meetings or for exploratory analysis.  
Does not affect the batch — modify freely.


In [ ]:
# if MEASURE == "raw":
#     plot_compare_condition(
#         stim      = "deviant",
#         component = "P3-early",
#         cluster   = ["FC1", "FCz", "FC2", "C1", "Cz", "C2"],
#         windows   = ((180, 260),),
#         ylim      = dict(eeg=[-8, 8]),
#     )